# S10: Transformaciones de variables

Daniel Pallares

## Paso 0: Cargar el dataset codificado de la Semana 9

`S09_PD_PallaresArbelaezSanchez_DatasetCodificado.csv` todavía vive en la carpeta `S09/`, no se movió, así que lo cargo con una ruta relativa en vez de copiarlo de nuevo a esta carpeta.

In [1]:
import pandas as pd

df = pd.read_csv("../S09/S09_PD_PallaresArbelaezSanchez_DatasetCodificado.csv")
df.shape

(551790, 44)

## Variables temporales a partir de `fecha_pedido`

Al leer un CSV, pandas no conserva el tipo `datetime64[ns]`. `fecha_pedido` vuelve como texto. Confirmo el tipo y la convierto antes de extraer cualquier variable temporal.

In [2]:
df["fecha_pedido"].dtype

dtype('O')

In [3]:
df["fecha_pedido"] = pd.to_datetime(df["fecha_pedido"], errors="coerce")
df["fecha_pedido"].dtype, df["fecha_pedido"].isnull().sum()

(dtype('<M8[ns]'), np.int64(19583))

`errors="coerce"` convierte a `NaT` las fechas que no se puedan interpretar. Ya había filas con `fecha_pedido` vacío desde semanas anteriores, así que esas quedan en `NaT` y sus variables temporales derivadas quedan en blanco. Es un dato faltante conocido, no algo nuevo que introduzca esta celda.

In [4]:
df["mes_pedido"] = df["fecha_pedido"].dt.month
df["trimestre_pedido"] = df["fecha_pedido"].dt.quarter
df["dia_semana_pedido"] = df["fecha_pedido"].dt.day_name()
df["es_fin_semana_pedido"] = df["fecha_pedido"].dt.dayofweek >= 5

df[["fecha_pedido", "mes_pedido", "trimestre_pedido", "dia_semana_pedido", "es_fin_semana_pedido"]].head()

,fecha_pedido,mes_pedido,trimestre_pedido,dia_semana_pedido,es_fin_semana_pedido
0,2025-11-15,11.0,4.0,Saturday,True
1,2026-03-05,3.0,1.0,Thursday,False
2,2025-06-27,6.0,2.0,Friday,False
3,2025-05-01,5.0,2.0,Thursday,False
4,2025-08-29,8.0,3.0,Friday,False


## Variables de texto a partir de `comentario_cliente`

In [5]:
df["comentario_cliente"] = df["comentario_cliente"].fillna("")
df["comentario_cliente"].isnull().sum()

np.int64(0)

In [6]:
df["longitud_comentario"] = df["comentario_cliente"].str.len()
df["num_palabras_comentario"] = df["comentario_cliente"].str.split().str.len()
df["tiene_comentario"] = ~df["comentario_cliente"].isin(["", "Sin comentario"])
df["menciona_demora"] = df["comentario_cliente"].str.contains("demor|tard|retras", case=False, na=False)

df[["comentario_cliente", "longitud_comentario", "num_palabras_comentario", "tiene_comentario", "menciona_demora"]].head()

,comentario_cliente,longitud_comentario,num_palabras_comentario,tiene_comentario,menciona_demora
0,El domiciliario fue muy amable,30,5,True,False
1,El producto llegó dañado,24,4,True,False
2,Rápido y sin problemas,22,4,True,False
3,Sin comentario,14,2,False,False
4,Llegó incompleto el pedido,26,4,True,False


`tiene_comentario` es `False` cuando el comentario quedó vacío o es el placeholder `"Sin comentario"` que dejaron las semanas de limpieza de datos faltantes. `menciona_demora` busca palabras relacionadas con demoras, como demora, tarde o retraso, sin importar mayúsculas.

## Paso 1: Vuelve a separar entrenamiento y prueba

In [7]:
from sklearn.model_selection import train_test_split

df_entrenamiento, df_prueba = train_test_split(df, test_size=0.2, random_state=42)
len(df_entrenamiento), len(df_prueba), len(df)

(441432, 110358, 551790)

## Paso 2: Discretiza `edad_cliente` con bins fijos de negocio

Aplico `pd.cut()` sobre `df` completo, no sobre `df_entrenamiento`, porque los límites 17, 25, 35, 45, 60 y 100 son reglas fijas del negocio, no un cálculo aprendido de los propios datos. Por eso no hay riesgo de fuga de datos en este paso.

In [8]:
df["grupo_edad"] = pd.cut(
    df["edad_cliente"],
    bins=[17, 25, 35, 45, 60, 100],
    labels=["18 a 25", "26 a 35", "36 a 45", "46 a 60", "61 y más"],
)
df["grupo_edad"].value_counts(dropna=False).sort_index()

grupo_edad
18 a 25      71594
26 a 35      89606
36 a 45      89369
46 a 60     134949
61 y más    134314
NaN          31958
Name: count, dtype: int64

Hay clientes con `edad_cliente` fuera del rango 18 a 100, con edades negativas o mayores a 100. Es un problema de exactitud ya señalado desde la auditoría de calidad de la Semana 3. Esas filas quedan con `grupo_edad` en NaN porque caen fuera de los bins de negocio. No las fuerzo dentro de un grupo porque no sé cuál sería su edad real. Las dejo documentadas como observación en vez de inventar un valor.

## Paso 3: Vuelve a dividir para incluir la nueva columna en ambos conjuntos

Repito la división usando el mismo `random_state=42`. Quedan las mismas filas en entrenamiento y en prueba que en el Paso 1, solo que ahora `df` ya incluye `grupo_edad` en ambos conjuntos.

In [9]:
df_entrenamiento, df_prueba = train_test_split(df, test_size=0.2, random_state=42)
"grupo_edad" in df_entrenamiento.columns, "grupo_edad" in df_prueba.columns

(True, True)

## Paso 4: Revisa el sesgo de `monto_compra` y aplica la transformación logarítmica

In [10]:
sesgo_original = df["monto_compra"].skew()
sesgo_original

np.float64(2.042959115668505)

El logaritmo no aprende nada de los datos, no ajusta ningún parámetro a partir de ellos, así que aplicarlo sobre el dataset completo antes o después de dividir da exactamente el mismo resultado fila por fila y no genera ningún riesgo de fuga de datos. Por eso lo aplico directamente sobre `df_entrenamiento` y `df_prueba`, que equivale a aplicarlo sobre `df` completo y dividir después.

In [11]:
import numpy as np

df_entrenamiento["monto_compra_log"] = np.log1p(df_entrenamiento["monto_compra"])
df_prueba["monto_compra_log"] = np.log1p(df_prueba["monto_compra"])

df_entrenamiento["monto_compra_log"].skew()

np.float64(-0.20245319884029103)

## Paso 5: Ajusta y aplica Box-Cox, solo con el conjunto de entrenamiento

A diferencia del logaritmo, `PowerTransformer(method="box-cox")` sí aprende un parámetro lambda a partir de la distribución de los datos que se le pasan en `fit()`. Si lo ajustara con el dataset completo, la prueba influiría en ese parámetro aprendido y se filtraría información de la prueba hacia el entrenamiento, generando fuga de datos. Por eso lo ajusto solo con `df_entrenamiento` y reutilizo ese mismo ajuste para transformar la prueba.

In [12]:
from sklearn.preprocessing import PowerTransformer

transformador_boxcox = PowerTransformer(method="box-cox")
transformador_boxcox.fit(df_entrenamiento[["monto_compra"]])

PowerTransformer(method='box-cox')

In [13]:
df_entrenamiento["monto_compra_boxcox"] = transformador_boxcox.transform(df_entrenamiento[["monto_compra"]])
df_prueba["monto_compra_boxcox"] = transformador_boxcox.transform(df_prueba[["monto_compra"]])

df_entrenamiento["monto_compra_boxcox"].skew()

np.float64(-0.005346100418620022)

## Paso 6: Ensambla, compara y exporta tu dataset final de la semana

In [14]:
df_final = pd.concat([df_entrenamiento, df_prueba]).sort_index()
df_final.shape

(551790, 55)

In [15]:
comparacion_sesgo = pd.DataFrame({
    "variable": ["monto_compra", "monto_compra_log", "monto_compra_boxcox"],
    "sesgo": [
        df_final["monto_compra"].skew(),
        df_final["monto_compra_log"].skew(),
        df_final["monto_compra_boxcox"].skew(),
    ],
})
comparacion_sesgo

,variable,sesgo
0,monto_compra,2.042959
1,monto_compra_log,-0.199733
2,monto_compra_boxcox,-0.002856


## Comparación de sesgo antes y después

`monto_compra` original tiene un sesgo positivo marcado, con una cola larga hacia montos altos. El logaritmo, en `monto_compra_log`, ya lo reduce bastante, pero Box-Cox, en `monto_compra_boxcox`, es el que más lo acerca a 0, porque busca automáticamente el parámetro lambda que mejor normaliza esta distribución específica, en vez de usar una transformación fija como el logaritmo. La tabla de la celda anterior, `comparacion_sesgo`, muestra los tres valores lado a lado.

In [16]:
df_final.to_csv("S10_PD_Pallares_DatasetTransformado.csv", index=False)
pd.read_csv("S10_PD_Pallares_DatasetTransformado.csv", nrows=1).shape

(1, 55)